In [ ]:
### HERE COMES THE ENITRE ANALYSIS AND THE PLOTS I WILL CREATE
### IT WILL ACCESS THE SAME FUNCTIONS (PARTIALLY) AS THE 4_homologs_analysis_visualization.ipynb

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path for custom src imports later
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Default plot style
# sns.set_theme(style="whitegrid", context="notebook")
# plt.rcParams["figure.dpi"] = 100

In [ ]:
DATA_DIR = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"

df = pd.read_parquet(f"{DATA_DIR}/variants_annotated_final.parquet")
print(f"Loaded {len(df):,} variant-region assignments")
print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")

# Also load the regions JSON — useful for context (WT sequences, etc.)
with open(f"{DATA_DIR}/genomic_coords_merged_win5.json") as f:
    regions = json.load(f)
region_by_id = {r["region_id"]: r for r in regions}
print(f"Loaded {len(regions)} regions from JSON")

In [ ]:
import json
with open(f"{DATA_DIR}/genomic_coords_merged_win5.json") as fh:
    new_regions = json.load(fh)

# Check convention consistency
conventions = []
for r in new_regions:
    s, e = r["prot_region"]
    L = len(r["prot_seq"])
    if L == e - s:
        conventions.append("half_open")
    elif L == e - s + 1:
        conventions.append("inclusive")
    else:
        conventions.append(f"broken (seq_len {L}, range {e-s})")

from collections import Counter
print(Counter(conventions))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SANITY CHECK: coordinate convention should now be uniform (0-based half-open)
# This is just a verification — the new coding_dna_v2 pipeline produces
# consistent output by construction. No patching needed.
# ═══════════════════════════════════════════════════════════════════════════

inconsistent = []
for r in regions:
    start, end = r["prot_region"]
    seq_len = len(r["prot_seq"])
    if seq_len != end - start:
        inconsistent.append((r["region_id"], start, end, seq_len))

if inconsistent:
    raise RuntimeError(
        f"Found {len(inconsistent)} regions with inconsistent coord convention: "
        f"{inconsistent[:5]}. Re-check coding_dna_v2 output."
    )
print(f"Coordinate convention check passed: all {len(regions)} regions are 0-based half-open.")


# ═══════════════════════════════════════════════════════════════════════════
# WT-MATCH CHECK: variant ref AA from VEP should match prot_seq at that position
# This catches VEP/MANE inconsistencies, not coordinate bugs.
# Keep this regardless of coordinate method.
# ═══════════════════════════════════════════════════════════════════════════

def _check(row):
    region = region_by_id.get(row["region_id"])
    if region is None or pd.isna(row["protein_position_int"]):
        return None
    # Note: prot_region is 0-based half-open, so position offset is:
    pos = int(row["protein_position_int"]) - int(row["region_start_aa"]) - 1
    # The -1 accounts for protein_position_int being 1-based (VEP convention),
    # while region_start_aa is 0-based (our convention).
    if pos < 0 or pos >= len(region["prot_seq"]):
        return None
    if row["before_aa"] is None or row["before_aa"] == "-" or len(row["before_aa"]) != 1:
        return None
    return region["prot_seq"][pos] == row["before_aa"]

df["wt_match"] = df.apply(_check, axis=1)
print(df["wt_match"].value_counts(dropna=False))

df["rg_analysis_reliable"] = df["wt_match"] == True
df_for_rg = df[df["rg_analysis_reliable"]].copy()

print(f"\nTotal variants: {len(df):,}")
print(f"Usable for RG analysis: {len(df_for_rg):,}")
print(f"\nBreakdown by group:")
print(df_for_rg["group"].value_counts())

In [ ]:
# Per-group region count and total motif coverage in AAs
for group in ["pos", "neg"]:
    grp_regions = [r for r in regions if r["group"] == group]
    n = len(grp_regions)
    total_aa = sum(r["prot_region"][1] - r["prot_region"][0] for r in grp_regions)
    n_var = (df_for_rg["group"] == group).sum()
    print(f"{group}: {n} regions, {total_aa} AAs total, "
          f"{n_var} variants ({n_var/total_aa:.2f} variants/AA)")

In [ ]:
###### NOW ANALYSIS

In [ ]:
import src.analysis_visualization.region_analysis as ra

fig, stats = ra.plot_variant_density(df_for_rg, dataset="gnomad")
plt.show()

In [ ]:
fig, results = ra.plot_consequence_distributions(df_for_rg, dataset="gnomad")
plt.show()

In [ ]:
fig, r = ra.plot_median_alphamissense(df_for_rg, dataset="gnomad")
plt.show()

In [ ]:
# %%
%load_ext autoreload
%autoreload 2

from pathlib import Path
from src.analysis_visualization.esm_llr import (
    download_esm_llr_zip,
    annotate_variants_with_esm,
    run_esm_analysis,
    run_esm_analysis_af_stratified,
)

# %%
# One-time download (~1.34 GB, runs from your notebook)
ZIP_PATH = Path("data/esm1b/ALL_hum_isoforms_ESM1b_LLR.zip")
download_esm_llr_zip(ZIP_PATH)

# %%
# Quick sanity check: can we load one of your proteins?
from src.analysis_visualization.esm_llr import load_esm_scores_for_protein
test_uid = df["uniprot_accession"].dropna().iloc[0]
test = load_esm_scores_for_protein(test_uid, ZIP_PATH)
print(f"Test protein {test_uid}:")
print(test.head() if test is not None else "  NOT FOUND in catalog")

# %%
# Annotate
df_esm = annotate_variants_with_esm(
    df,
    zip_path=ZIP_PATH,
    uniprot_col="uniprot_accession",
    pos_col="Protein_position",
    after_col="after_aa",
    consequence_col="Consequence",
)
df_esm.to_parquet("data/processed/variants_with_esm.parquet")

# %%
result = run_esm_analysis(df_esm, annotate=False, dataset="gnomad")
# result_af = run_esm_analysis_af_stratified(df_esm, dataset="gnomad")

In [ ]:
import src.analysis_visualization.rg_analysis as rga
df_rg = rga.compute_rg_disruption_columns(df_for_rg, region_by_id)
print("RG hit columns added. Summary:")
print(df_rg["hits_rg"].value_counts())

# Run each plot independently
fig_a, r_a = rga.plot_region_length(region_by_id, dataset="gnomad")
plt.show()

fig_b, r_b = rga.plot_n_rg_motifs(region_by_id, dataset="gnomad")
plt.show()

fig_c, r_c = rga.plot_rg_density(region_by_id, dataset="gnomad")
plt.show()

# D1 — density version (replaces the saturating D)
fig_d1, r_d1 = rga.plot_variants_per_rg_by_type(df_rg, region_by_id, dataset="gnomad")
plt.show()

# # D2 — AlphaMissense on RG-hitting missense
# fig_d2, r_d2 = rga.plot_median_alphamissense_on_rgs(df_rg, dataset="gnomad")
# plt.show()


In [ ]:
fig, r = rga.plot_rg_role_asymmetry(df_rg, dataset="gnomad")
plt.show()

In [ ]:
results = rga.plot_rg_change_events_stacked(df_rg, region_by_id, dataset="gnomad")

# the single event version has been commmented out in the rg-analysis, because it showed no signifincance in any plot, so here is just the stacked version

In [ ]:
# Make sure df_events is computed (already have compute_rg_change_events from before)
df_events = rga.compute_rg_change_events(df_rg, region_by_id)

# Analysis 1
loss_transition_results = rga.plot_rg_loss_transitions(df_events, dataset="gnomad")
plt.show()

# Analysis 2
cluster_results = rga.plot_isolated_vs_clustered_loss(
    df_events, region_by_id,
    window_sizes=[2,4,6],
    dataset="gnomad",
)
plt.show()

In [ ]:
gain_transition_results = rga.plot_rg_gain_transitions(df_events, dataset="gnomad")
plt.show()

In [ ]:
# Build the null once — reused for both plots
null_results = rga.build_enumeration_null(region_by_id, df_for_rg)

# RG events observed vs expected
rg_comparison = rga.plot_rg_events_observed_vs_expected(
    df_events, null_results, dataset="gnomad",
)
plt.show()

# Consequences observed vs expected
cons_comparison = rga.plot_consequences_observed_vs_expected(
    df_for_rg, null_results, dataset="gnomad",
)
plt.show()

In [ ]:
# Build null once
null_results = rga.build_enumeration_null(region_by_id, df_for_rg)

# RG events: 4 subplots (no_change / loss / gain / movement)
rg_results = rga.plot_rg_events_vs_expected_boxes(
    df_events, null_results, dataset="gnomad",
)
plt.show()

# Consequences: 3 subplots (synonymous / missense / nonsense)
cons_results = rga.plot_consequences_vs_expected_boxes(
    df_for_rg, null_results, dataset="gnomad",
)
plt.show()

In [ ]:
# Plot A: per-variant distribution, R/G-affecting only
fig_a, r_a = rga.plot_delta_rg_ratio_per_variant(df_rg, region_by_id, dataset="gnomad")
plt.show()

# Plot B: per-region mean across all missense
fig_b, r_b = rga.plot_delta_rg_ratio_per_region(df_rg, region_by_id, dataset="gnomad")
plt.show()

In [ ]:
############### PHYSCHEM

In [ ]:
import src.analysis_visualization.physchem_analysis as pca

# # Step 1: compute per-variant deltas (slow — uses multiprocessing)
# deltas_df = pca.compute_physchem_deltas(df_rg, region_by_id)

# # Save the result — expensive to recompute
# deltas_df.to_parquet(
#     "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/physchem_deltas.parquet"
# )

deltas_df = pd.read_parquet("/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/physchem_deltas.parquet"
)

# Step 2: aggregate to per-region means for classifier features
per_region = pca.aggregate_per_region(deltas_df)

# Step 3: plot each feature individually
all_results = pca.plot_all_delta_features(per_region, dataset="gnomad")

# Bonus: per-region WT baseline features (no variants involved — useful for classifier too)
wt_features = pca.compute_wt_physchem(region_by_id)

In [ ]:
################# AMINO ACID SUBSTITUTIONS

In [ ]:
import src.analysis_visualization.substitution_matrix_analysis as smx

raw   = smx.run_substitution_analysis(df_for_rg, dataset="gnomad", min_total=5)
comp  = smx.run_composition_normalized_analysis(df_for_rg, region_by_id, dataset="gnomad")
mut   = smx.run_mutability_normalized_analysis(df_for_rg, region_by_id, dataset="gnomad")

In [ ]:
mut = smx.run_mutability_normalized_analysis(df_for_rg, region_by_id, dataset="gnomad", save=True, save_table_path="/mnt/d/phd/scripts/16_ev_signature_predictor/data/output/gnomad_mutability_substitution_table.csv")

# # all points
# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut", show_significance=True, bg_span=2.0)

# # R-highlighted version
# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_r", highlight_sources=["R"],
#                          show_significance=True, bg_span=2.0)

# # G-highlighted version
# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_g", highlight_sources=["G"],
#                          show_significance=True, bg_span=2.0)

# # Y-highlighted version
# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_y", highlight_sources=["Y"],
                        #  show_significance=True, bg_span=2.0)

a, b = smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=["C", "G", "P"],
                         show_significance=False, bg_span=2.0)
a
# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=["A", "V", "I", "L", "M"],
#                          show_significance=False, bg_span=2.0)

# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=list("STNQ"),
#                          show_significance=False, bg_span=2.0)

# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=list("RKH"),
#                          show_significance=False, bg_span=2.0)

# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=list("DE"),
#                          show_significance=False, bg_span=2.0)

# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=list("FWY"),
#                          show_significance=False, bg_span=2.0)

# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=["A"],
#                         show_significance=False, bg_span=2.0)

# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=["V"],
#                         show_significance=False, bg_span=2.0)
# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=["I"],
#                         show_significance=False, bg_span=2.0)
# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=["L"],
#                         show_significance=False, bg_span=2.0)
# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=["M"],
#                         show_significance=False, bg_span=2.0)

# smx.plot_obs_exp_scatter(mut, dataset="gnomad_raw", n_label=0, show_significance=True) #n_label=200,always_label_sources=["R","G"]
# smx.plot_obs_exp_scatter(raw, dataset="gnomad_mutability", show_significance=True, always_label_sources=["R","G"])

In [ ]:
print(0 or 12)

In [ ]:
rates = smx.load_mutation_rates()


grp_result = smx.compute_grouped_substitution_matrix(df_for_rg, region_by_id, rates)

mut = smx.run_mutability_normalized_analysis(df_for_rg, region_by_id, dataset="gnomad")

# all points (use as supplement, or with label_all=False)
a, b = smx.plot_obs_exp_scatter_grouped(mut, dataset="gnomad_mut_all",
                                 show_significance=True, bg_span=1.5, label_all=False, n_label=0)
a
# # highlighted, readable main version
# smx.plot_obs_exp_scatter_grouped(mut, dataset="gnomad_mut_pos",
#                                  highlight_sources=["Pos"], show_significance=True, bg_span=1.5)
# smx.plot_obs_exp_scatter_grouped(mut, dataset="gnomad_mut_cgp",
#                                  highlight_sources=["C/G/P"], show_significance=True, bg_span=1.5)
# smx.plot_obs_exp_scatter_grouped(mut, dataset="gnomad_mut_neg",
#                                  highlight_sources=["Neg"], show_significance=True, bg_span=1.5)
# smx.plot_obs_exp_scatter_grouped(mut, dataset="gnomad_mut_polar",
#                                  highlight_sources=["Polar"], show_significance=True, bg_span=1.5)
# smx.plot_obs_exp_scatter_grouped(mut, dataset="gnomad_mut_aromatic",
#                                  highlight_sources=["Aromatic"], show_significance=True, bg_span=1.5)
# smx.plot_obs_exp_scatter_grouped(mut, dataset="gnomad_mut_hydrophobic",
#                                  highlight_sources=["Hydrophobic"], show_significance=True, bg_span=1.5)

In [ ]:


# grouped heatmap (obs/exp, binomial-tested)
grp = smx.run_grouped_substitution_analysis(df_for_rg, region_by_id, dataset="gnomad_mut")

# grouped scatter — reuses the 20x20 mutability result, swappable grouping
mut = smx.run_mutability_normalized_analysis(df_for_rg, region_by_id, dataset="gnomad")
smx.plot_obs_exp_scatter_grouped(mut, dataset="gnomad_mut")
# smx.plot_obs_exp_scatter_grouped(mut, groups=MY_CHARGE_GROUPING, dataset="gnomad_charge")  # brainstorm

In [ ]:
df_for_rg = smx.annotate_cpg_status(df_for_rg, "/mnt/d/phd/scripts/16_ev_signature_predictor/data/refs/GRCh38.primary_assembly.genome.fa")   # once

rates = smx.load_mutation_rates()
allc  = smx.compute_composition_normalized_enrichment(df_for_rg, region_by_id, rates=rates, cpg_filter="all")
nonc  = smx.compute_composition_normalized_enrichment(df_for_rg, region_by_id, rates=rates, cpg_filter="noncpg")
cpg   = smx.compute_composition_normalized_enrichment(df_for_rg, region_by_id, rates=rates, cpg_filter="cpg")

smx.plot_obs_exp_scatter_grouped(nonc, dataset="gnomad_noncpg")   # the decisive one
# smx.plot_obs_exp_scatter_grouped(cpg,  dataset="gnomad_cpg")
# smx.plot_obs_exp_scatter(nonc, dataset="gnomad_noncpg",n_label= 0, show_significance=True, always_label_sources=["R"])
# smx.plot_obs_exp_scatter(cpg, dataset="gnomad_cpg", show_significance=True, always_label_sources=["R","G"])

In [ ]:
import src.analysis_visualization.substitution_matrix_analysis as smx

# All-missense matrix (single call)
enrichment = smx.run_substitution_analysis(
    df_for_rg, dataset="gnomad", min_total=5,
)

# Access individual outputs
log2_or_matrix = enrichment["log2_or"]
fdr_matrix = enrichment["fdr"]
counts_pos = enrichment["counts_pos"]

In [ ]:
import src.analysis_visualization.substitution_matrix_analysis as smx

# Side-by-side rare (AF < 1e-4) vs common (AF >= 1e-3) matrices
results = smx.plot_af_comparison_matrices(
    df_rg,
    af_rare_max=1e-5,
    af_common_min=1e-5,
    dataset="gnomad",
)

# Access individual results
rare_enrichment = results["rare"]
common_enrichment = results["common"]

In [ ]:
# %%
%autoreload 2
from src.analysis_visualization.substitution_matrix_analysis import (
    run_marginal_substitution_analysis,
    SNP_REACHABLE,  # optionally inspect the dict
)

# Sanity check the reachability dict
print("R can reach:", SNP_REACHABLE["R"])
print("W can reach:", SNP_REACHABLE["W"])

# # %%
# result_marginal = run_marginal_substitution_analysis(
#     df,
#     dataset="gnomad",
#     min_total=1,
# )


# Supplementary — all tested AAs
result_marginal = run_marginal_substitution_analysis(df, dataset="gnomad")

# Publication figure — significant AAs only, auto-selected
run_marginal_substitution_analysis(
    df, dataset="gnomad",
    sig_only=True,
    save=True,
)

# Or hand-pick specific AAs you want to highlight
run_marginal_substitution_analysis(
    df, dataset="gnomad",
    source_aas=["G", "P", "R"],
    save=True,
)


# # Just the heatmap
# run_marginal_substitution_analysis(df, dataset="gnomad", plot_kind="enrichment_heatmap", save=True)

# # Focused bars for the significant AAs only
# run_marginal_substitution_analysis(
#     df, dataset="gnomad", sig_only=True,
#     plot_kind="enrichment_bars", save=True,
# )

# # Everything
# run_marginal_substitution_analysis(df, dataset="gnomad", plot_kind="all", save=True)

In [ ]:
# Total G→X variants in common matrix, per group
r_common = results["common"]
g_row_pos = r_common["counts_pos"].loc["G"].sum()
g_row_neg = r_common["counts_neg"].loc["G"].sum()
print(f"G→anything: pos = {g_row_pos}, neg = {g_row_neg}")
print(f"G→S fraction: pos = {77/g_row_pos:.3f}, neg = {19/g_row_neg:.3f}")

In [ ]:
# Quick check
gs_pos = df_rg[
    (df_rg["before_aa"] == "G") & 
    (df_rg["after_aa"] == "S") & 
    (df_rg["group"] == "pos") &
    (df_rg["AF_joint"] >= 1e-5) &
    (df_rg["Consequence"].str.contains("missense_variant", na=False))
]
gs_neg = df_rg[
    (df_rg["before_aa"] == "G") & 
    (df_rg["after_aa"] == "S") & 
    (df_rg["group"] == "neg") &
    (df_rg["AF_joint"] >= 1e-5) &
    (df_rg["Consequence"].str.contains("missense_variant", na=False))
]
print("G→S codon distribution in pos:")
print(gs_pos["Codons"].value_counts().head(10))
print("\nG→S codon distribution in neg:")
print(gs_neg["Codons"].value_counts().head(10))

In [ ]:
#### add here the proportion of each amino acid how often it appears in the regions, so that i can maybe find something intersting

In [ ]:
import src.analysis_visualization.codon_usage as cu

df_comp, comp_stats = cu.compute_gc_cpg_by_group(region_by_id)
cu.plot_gc_cpg_by_group(df_comp, comp_stats, dataset="gnomad")

In [ ]:
rates = smx.load_mutation_rates()


merged, flux_stats, match_rate = cu.compute_gc_cpg_flux(
    df_for_rg, region_by_id, rates, cu.enumerate_single_nt_substitutions)
print("context match rate:", match_rate)   # must be ~1.0 to trust ΔCpG
cu.plot_gc_cpg_flux(merged, flux_stats, dataset="gnomad")

In [ ]:
import pandas as pd
mis = df_for_rg[df_for_rg["Consequence"].fillna("").str.contains("missense_variant")]

def diag(v, region_by_id, shift=0):
    rid = v["region_id"]
    if rid not in region_by_id: return None
    dna = region_by_id[rid]["dna"].upper()
    pc = cu._parse_codons(v.get("Codons"))
    if pc is None: return None
    _, _, off, cref, calt = pc
    ci = int(v["protein_position_int"]) - int(v["region_start_aa"]) + shift
    dpos = 3*ci + off
    if not (0 <= dpos < len(dna)): return ("oob", cref, None)
    return (dna[dpos] == cref, cref, dna[dpos])

# how well does each candidate shift match?
for shift in [-1, 0, 1]:
    res = [diag(v, region_by_id, shift) for _, v in mis.iterrows()]
    ok = sum(1 for r in res if r and r[0] is True)
    tot = sum(1 for r in res if r and r[0] in (True, False))
    print(f"shift={shift:+d}: match {ok}/{tot} = {ok/tot:.2%}" if tot else f"shift={shift}: no valid")

In [ ]:
import src.analysis_visualization.codon_usage as cu

rates = smx.load_mutation_rates()
df_cm, cm_stats, cm = cu.compute_codon_mutability_by_group(
    region_by_id, rates)  
# cu.plot_codon_mutability(df_cm, cm_stats, cm, amino_acids=list("ADEGL"), dataset="gnomad")
# cu.plot_codon_mutability(df_cm, cm_stats, cm, amino_acids=list("PQRSY"), dataset="gnomad")
# cu.plot_codon_mutability(df_cm, cm_stats, cm, amino_acids=list("AVILM"), dataset="gnomad")
# cu.plot_codon_mutability(df_cm, cm_stats, cm, amino_acids=list("CGP"), dataset="gnomad")
cu.plot_codon_mutability(df_cm, cm_stats, cm, amino_acids=list("GRPSA"), dataset="gnomad")
# cu.plot_codon_mutability(df_cm, cm_stats, cm, amino_acids=["R", "G", "P", "C"], dataset="gnomad")

In [ ]:
import src.analysis_visualization.codon_usage as cu

# Supplementary — full grid, all 18 AAs
out = cu.run_codon_usage_analysis(region_by_id, dataset="gnomad")

# Publication — only AAs you want to feature
# cu.plot_codon_usage(
#     out["codon_counts"], out["test_results"],
#     source_aas=list("ADEGLPQRSY"), ncols= 5,
#     dataset="gnomad",
# )

cu.plot_codon_usage(
    out["codon_counts"], out["test_results"],
    source_aas=list("GRPSA"), ncols= 1,
    dataset="gnomad",
)


# # Auto-select significant
# cu.plot_codon_usage(
#     out["codon_counts"], out["test_results"],
#     sig_only=True,
#     dataset="gnomad",
# )
# print('hello')


In [ ]:
 
result = smx.run_codon_substitution_analysis(
    df_rg
)

# def run_codon_substitution_analysis(
#     df: pd.DataFrame,
#     group_col: str = "group",
#     pos_label: str = "pos",
#     neg_label: str = "neg",
#     min_total: int = 5,
#     dataset: str = "gnomad",
#     save: bool = True,
#     **plot_kwargs,
# ) -> dict:
#     """
#     [Dataset-agnostic]
#     End-to-end codon substitution matrix: build counts, enrichment, plot.
#     """
#     df_pos = df[df[group_col] == pos_label]
#     df_neg = df[df[group_col] == neg_label]
 
#     counts_pos = compute_codon_substitution_counts(df_pos)
#     counts_neg = compute_codon_substitution_counts(df_neg)
 
#     enrichment = compute_codon_enrichment(counts_pos, counts_neg, min_total=min_total)
#     plot_codon_substitution_matrix(enrichment, dataset=dataset, save=save, **plot_kwargs)
 
#     return enrichment
 

In [ ]:
#### AF analysis

In [ ]:
import src.analysis_visualization.af_spectrum as afs

stats = afs.plot_af_spectrum_cdf(df_rg, dataset="gnomad")

In [ ]:
import src.analysis_visualization.af_spectrum as afs

# df_rg must have `is_rg_disrupting` column (from compute_rg_disruption_columns)
stats = afs.plot_af_spectrum_by_subset(df_rg, dataset="gnomad")

In [ ]:
# Quick check in notebook
rg = df_rg[df_rg["is_rg_disrupting"] & df_rg["AF_joint"].notna()]
for group in ["pos", "neg"]:
    g = rg[rg["group"] == group]["AF_joint"]
    common = g[g >= 1e-4]
    very_common = g[g >= 1e-3]
    print(f"{group}: n={len(g):,}, n≥1e-4={len(common)}, n≥1e-3={len(very_common)}, "
          f"max={g.max():.4f}")

In [ ]:
#$###### CLASSIFIER FEATURE COLLECTION

In [ ]:
df_esm

In [ ]:
import src.analysis_visualization.classifier_features as cf
# import reload
# reload(cf)
# If you have physchem_deltas computed, load it; otherwise pass None
try:
    physchem_deltas_df = pd.read_parquet(
        "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/physchem_deltas.parquet"
    )
except FileNotFoundError:
    physchem_deltas_df = None

features_df = cf.build_classifier_features(
    df_rg=df_rg,
    df_events=df_events,
    region_by_id=region_by_id,
    physchem_deltas_df=physchem_deltas_df,
    df_esm=df_esm,                  # ← pass the annotated df
)

# Save it
cf.save_features(features_df)



#### I WANT TO ADD THE AF AND ESM LLR

In [ ]:
features_df

In [ ]:
#### HERE SWITCH TO 3_RF_model_creation.ipynb

In [ ]:
for a in features_df.column

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Feature selection pipeline: correlation pruning + RFECV with grouped CV
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# ── Config ──────────────────────────────────────────────────────────────
RANDOM_STATE = 42
N_TREES = 100
CORR_THRESHOLD = 0.95
N_SPLITS = 5

POS_COLOR = "#4daf4a"
NEG_COLOR = "#e41a1c"

GROUP_COL = "uniprot_accession"   # ← change to your actual protein/cluster column

EXCLUDE_COLS = {"region_id", "group", "label", "group_x", "group_y", "uniprot_accession"}

AM_FEATURES = [c for c in features_df.columns
               if "alphamissense" in c.lower() or c.startswith("am_")
               or c == "fraction_pathogenic"]

# ── Prep ────────────────────────────────────────────────────────────────
all_features = [c for c in features_df.columns if c not in EXCLUDE_COLS]
numeric_features = features_df[all_features].select_dtypes(include=[np.number]).columns.tolist()
dropped_nonnumeric = set(all_features) - set(numeric_features)
if dropped_nonnumeric:
    print(f"Dropped {len(dropped_nonnumeric)} non-numeric features: {sorted(dropped_nonnumeric)}")

X_df = features_df[numeric_features].copy()
y = (features_df["group"] == "pos").astype(int).values
groups = features_df[GROUP_COL].values

print(f"Starting with {X_df.shape[1]} numeric features, "
      f"{X_df.shape[0]} regions, {len(np.unique(groups))} unique groups")

# ── Step 1: Drop near-zero-variance features ───────────────────────────
variances = X_df.var()
near_zero = variances[variances < 1e-8].index.tolist()
if near_zero:
    print(f"\n[Step 1] Dropping {len(near_zero)} near-zero-variance features: {near_zero}")
    X_df = X_df.drop(columns=near_zero)

# ── Step 2: Correlation pruning ─────────────────────────────────────────
def prune_correlated(df, threshold=0.95):
    """Drop one of each highly correlated pair; keep the higher-variance one."""
    corr = df.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = set()
    for col in upper.columns:
        if col in to_drop:
            continue
        correlated = upper.index[upper[col] > threshold].tolist()
        for partner in correlated:
            if partner in to_drop:
                continue
            # Keep whichever has more variance
            loser = partner if df[col].var() >= df[partner].var() else col
            to_drop.add(loser)
    return [c for c in df.columns if c not in to_drop], sorted(to_drop)

kept, pruned = prune_correlated(X_df, threshold=CORR_THRESHOLD)
print(f"\n[Step 2] Correlation pruning at |r| > {CORR_THRESHOLD}:")
print(f"  Kept: {len(kept)}    Dropped: {len(pruned)}")
if pruned:
    print(f"  Dropped features: {pruned}")
X_df = X_df[kept]

# ── Step 3: Baseline CV AUC on pruned set ──────────────────────────────
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
baseline_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1)),
])
baseline_auc = cross_val_score(baseline_pipe, X_df.values, y, groups=groups,
                                cv=cv, scoring="roc_auc", n_jobs=-1)
print(f"\n[Step 3] Baseline CV AUC after pruning: "
      f"{baseline_auc.mean():.3f} ± {baseline_auc.std():.3f}")

# ── Step 4: RFECV with grouped CV ──────────────────────────────────────
print(f"\n[Step 4] Running RFECV (this may take a few minutes)...")

# Impute once for RFECV (it doesn't accept a pipeline as estimator)
imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X_df.values)

rfecv = RFECV(
    estimator=RandomForestClassifier(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1),
    step=5,
    min_features_to_select=5,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
)
rfecv.fit(X_imputed, y, groups=groups)

selected_features = [f for f, keep in zip(X_df.columns, rfecv.support_) if keep]
print(f"  RFECV selected {rfecv.n_features_} features (out of {X_df.shape[1]})")
print(f"  CV AUC at optimum: {rfecv.cv_results_['mean_test_score'][rfecv.n_features_ // 5]:.3f}")

# ── Step 5: Visualize ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: AUC vs number of features
ax = axes[0]
n_features_grid = np.arange(1, len(rfecv.cv_results_["mean_test_score"]) + 1) * 5
n_features_grid = n_features_grid[:len(rfecv.cv_results_["mean_test_score"])]
mean_scores = rfecv.cv_results_["mean_test_score"]
std_scores = rfecv.cv_results_["std_test_score"]

ax.plot(n_features_grid, mean_scores, "-o", color=POS_COLOR, markersize=4, linewidth=1.2)
ax.fill_between(n_features_grid, mean_scores - std_scores, mean_scores + std_scores,
                color=POS_COLOR, alpha=0.2)
ax.axvline(rfecv.n_features_, color=NEG_COLOR, linestyle="--", linewidth=1,
           label=f"Optimum: {rfecv.n_features_} features")
ax.set_xlabel("Number of features")
ax.set_ylabel("CV AUC (mean ± std)")
ax.set_title("RFECV: AUC vs feature count")
ax.legend(frameon=False)
ax.grid(alpha=0.3, linestyle=":", linewidth=0.4)

# Right: which AM features survived
ax = axes[1]
am_in_selected = [f for f in selected_features if f in AM_FEATURES]
am_dropped = [f for f in AM_FEATURES if f not in selected_features and f in X_df.columns]
categories = ["AM kept", "AM dropped", "Non-AM kept"]
counts = [len(am_in_selected), len(am_dropped), len(selected_features) - len(am_in_selected)]
colors = ["tab:orange", "lightgray", "tab:blue"]
ax.bar(categories, counts, color=colors, edgecolor="black", linewidth=0.4)
for i, c in enumerate(counts):
    ax.text(i, c + 0.3, str(c), ha="center", fontsize=10)
ax.set_ylabel("Count")
ax.set_title("Feature survival by group")
ax.grid(axis="y", alpha=0.3, linestyle=":", linewidth=0.4)

for ax in axes:
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

plt.tight_layout()
plt.show()

# ── Step 6: Final comparison ───────────────────────────────────────────
final_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1)),
])
final_auc = cross_val_score(final_pipe, X_df[selected_features].values, y, groups=groups,
                             cv=cv, scoring="roc_auc", n_jobs=-1)

print("\n" + "═" * 65)
print("SUMMARY")
print("═" * 65)
print(f"{'Stage':<35} {'N features':>12} {'CV AUC':>15}")
print("─" * 65)
print(f"{'Original':<35} {len(numeric_features):>12} {'(see prev run)':>15}")
print(f"{'After correlation pruning':<35} {len(kept):>12} "
      f"{baseline_auc.mean():.3f} ± {baseline_auc.std():.3f}")
print(f"{'After RFECV':<35} {rfecv.n_features_:>12} "
      f"{final_auc.mean():.3f} ± {final_auc.std():.3f}")

print(f"\nSelected features: ({len(kept)}):")
for f in kept:
    print(f"  • {f}")

In [ ]:
FEATURE_GROUPS = {
    # ── Sequence properties (region-level, no variants involved) ──────────
    "sequence_basic": [
        "region_length",
        "n_rg_motifs",
        "rg_fraction",
    ],

    # ── Variant burden: raw counts and overall density ────────────────────
    # "variant_burden": [
    #     "n_LoF",
    #     "n_inframe_indel",
    #     "n_other",
    #     "n_synonymous",
    #     "n_variants_total",
    #     "variant_density",
    # ],

    # ── Variant composition: per-consequence densities and fractions ──────
    "variant_composition": [
        "density_synonymous", "fraction_synonymous",
        "density_missense",   "fraction_missense",
        "density_inframe_indel", "fraction_inframe_indel",
        "density_LoF",        "fraction_LoF",
        # "n_LoF",
        # "n_inframe_indel",
        "n_other",
        # "n_synonymous",
        "n_variants_total",
        "variant_density",
    ],

    # ── AlphaMissense pathogenicity ───────────────────────────────────────
    "alphamissense": [
        "am_max",
        "am_std",
        # "am_fraction_pathogenic",   # not in current feature list
    ],

    # ── Allele frequency: gnomAD AF distribution per consequence class ────
    "allele_frequency": [
        # Variant counts per consequence (AF-derived)
        "af_n_syn", "af_n_indel", "af_n_lof",
        # Per-consequence AF summaries
        "af_median_log10_syn",
        "af_frac_singleton_syn", "af_frac_common_syn",
        "af_median_log10_mis",
        "af_frac_singleton_mis", "af_frac_common_mis",
        "af_median_log10_indel",
        "af_frac_singleton_indel", "af_frac_common_indel",
        "af_median_log10_lof",
        "af_frac_singleton_lof",
        # Cross-consequence ratios (selection signals)
        "af_log2ratio_mis_syn_count",
        "af_log2ratio_lof_syn_count",
        "af_median_log10_delta_mis_syn",
        "af_frac_rare_delta_mis_syn",
        # AM-weighted by rarity
        "af_am_score_weighted_by_rarity",
        # RG-restricted AF features (move to rg_events if you prefer)
        "af_rg_n_variants",
        "af_rg_median_log10",
        "af_rg_frac_singleton",
        "af_rg_vs_nonrg_log10_delta_mis",
    ],

    # ── ESM1b protein language model LLR ──────────────────────────────────
    "esm": [
        "esm_median",
        "esm_min",
        "esm_std",
        "esm_fraction_disruptive",
    ],

    # ── RG motif burden: how often RG motifs get hit, and by what ─────────
    # "rg_burden": [
    #     "rg_fraction_rgs_hit_LoF",
    #     "rg_fraction_rgs_hit_inframe_indel",
    #     "rg_fraction_rgs_hit_missense",
    #     "rg_fraction_rgs_hit_synonymous",
    #     "rg_mean_burden_on_hit_LoF",
    #     "rg_mean_burden_on_hit_inframe_indel",
    #     "rg_mean_burden_on_hit_missense",
    #     "rg_mean_burden_on_hit_synonymous",
    #     "n_g_hits_disrupting",
    #     "n_r_hits_disrupting",
    #     "rg_r_fraction",
    # ],

    # ── RG change events: gain/loss/no-change dynamics + RG burden ────────
    "rg_events": [
        "rg_event_fraction_no_change",
        "rg_event_fraction_gain",
        "rg_event_fraction_movement",
        "delta_rg_ratio_rel_mean",
        "rg_fraction_rgs_hit_LoF",
        "rg_fraction_rgs_hit_inframe_indel",
        "rg_fraction_rgs_hit_missense",
        "rg_fraction_rgs_hit_synonymous",
        "rg_mean_burden_on_hit_LoF",
        "rg_mean_burden_on_hit_inframe_indel",
        "rg_mean_burden_on_hit_missense",
        "rg_mean_burden_on_hit_synonymous",
        "n_g_hits_disrupting",
        "n_r_hits_disrupting",
        "rg_r_fraction",
    ],

    # ── Substitution biochemistry: rates of biochemical changes ───────────
    "substitution_classes": [
        "sub_rate_charge_alter",
        "sub_rate_pos_to_anything",
        "sub_rate_aromatic_alter",
        "sub_rate_hydrophobic_alter",
        "sub_rate_polar_alter",
        "sub_rate_proline_intro",
        "sub_rate_glycine_intro",
        "sub_rate_conservative",
    ],

    # ── Physchem (deltas + WT baseline) ───────────────────────────────────
    "physchem_deltas": [
        "delta_ncpr",
        "delta_fcr",
        "delta_kappa",
        "delta_hydropathy",
        "delta_aromaticity",
        "delta_fraction_proline",
        "delta_n_pos",
        "delta_n_neg",
        "wt_ncpr",
        "wt_fcr",
        "wt_kappa",
        "wt_hydropathy",
        "wt_aromaticity",
        "wt_fraction_proline",
        "wt_n_pos",
        "wt_n_neg",
    ],

    # # ── WT physchem: baseline sequence properties (no variants) ───────────
    # "physchem_wt": [
    #     "wt_ncpr",
    #     "wt_fcr",
    #     "wt_kappa",
    #     "wt_hydropathy",
    #     "wt_aromaticity",
    #     "wt_fraction_proline",
    #     "wt_n_pos",
    #     "wt_n_neg",
    # ],

    # ── Codon usage: synonymous codon preferences per AA ──────────────────
    "codon_usage": [
        "codon_A_GCT", "codon_A_GCC", "codon_A_GCA", "codon_A_GCG",
        "codon_C_TGT",
        "codon_D_GAT",
        "codon_E_GAG",
        "codon_F_TTC",
        "codon_G_GGT", "codon_G_GGC", "codon_G_GGA", "codon_G_GGG",
        "codon_H_CAC",
        "codon_I_ATT", "codon_I_ATC", "codon_I_ATA",
        "codon_K_AAG",
        "codon_L_TTA", "codon_L_TTG", "codon_L_CTT", "codon_L_CTC",
        "codon_L_CTA", "codon_L_CTG",
        "codon_N_AAC",
        "codon_P_CCT", "codon_P_CCC", "codon_P_CCA", "codon_P_CCG",
        "codon_Q_CAG",
        "codon_R_CGT", "codon_R_CGC", "codon_R_CGA", "codon_R_CGG",
        "codon_R_AGA", "codon_R_AGG",
        "codon_S_TCT", "codon_S_TCC", "codon_S_TCA", "codon_S_TCG",
        "codon_S_AGT", "codon_S_AGC",
        "codon_T_ACT", "codon_T_ACC", "codon_T_ACA", "codon_T_ACG",
        "codon_V_GTT", "codon_V_GTC", "codon_V_GTA", "codon_V_GTG",
        "codon_Y_TAC",
    ],
}

In [ ]:
# Re-run the prune step on features_df so it persists
all_features = [c for c in features_df.columns if c not in EXCLUDE_COLS]
numeric_features = features_df[all_features].select_dtypes(include=[np.number]).columns.tolist()

# Drop near-zero variance
X_check = features_df[numeric_features]
near_zero = X_check.var()[X_check.var() < 1e-8].index.tolist()
kept, pruned = prune_correlated(X_check.drop(columns=near_zero), threshold=0.95)

# Build the pruned df with metadata preserved
meta_cols = ["region_id", "uniprot_accession", "group", "label"]
meta_cols = [c for c in meta_cols if c in features_df.columns]
features_df_pruned = features_df[meta_cols + kept].copy()

print(f"Pruned df shape: {features_df_pruned.shape}")
print(f"Dropped {len(pruned)} correlated + {len(near_zero)} near-zero variance features")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Group-wise ROC: one curve per feature group, with reference "All features"
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_curve, auc

# ── Config ──────────────────────────────────────────────────────────────
RANDOM_STATE = 42
N_TREES = 100
N_SPLITS = 5
GROUP_COL = "uniprot_accession"

POS_COLOR = "#4daf4a"
NEG_COLOR = "#e41a1c"

# ── Prep ────────────────────────────────────────────────────────────────
y = (features_df_pruned["group"] == "pos").astype(int).values
groups_cv = features_df_pruned[GROUP_COL].values
# cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# def _cv_roc(X, y, groups_cv, cv, label):
#     """Run CV and return mean ROC curve + per-fold AUCs."""
#     pipe = Pipeline([
#         ("impute", SimpleImputer(strategy="median")),
#         ("rf", RandomForestClassifier(n_estimators=N_TREES,
#                                        random_state=RANDOM_STATE, n_jobs=-1)),
#     ])
#     mean_fpr = np.linspace(0, 1, 100)
#     tprs, aucs = [], []
#     for train_idx, test_idx in cv.split(X, y, groups=groups_cv):
#         pipe.fit(X[train_idx], y[train_idx])
#         y_proba = pipe.predict_proba(X[test_idx])[:, 1]
#         fpr, tpr, _ = roc_curve(y[test_idx], y_proba)
#         interp_tpr = np.interp(mean_fpr, fpr, tpr)
#         interp_tpr[0] = 0.0
#         tprs.append(interp_tpr)
#         aucs.append(auc(fpr, tpr))
#     mean_tpr = np.mean(tprs, axis=0)
#     mean_tpr[-1] = 1.0
#     std_tpr = np.std(tprs, axis=0)
#     return {
#         "label": label,
#         "mean_fpr": mean_fpr,
#         "mean_tpr": mean_tpr,
#         "std_tpr": std_tpr,
#         "auc_mean": np.mean(aucs),
#         "auc_std": np.std(aucs),
#         "n_features": X.shape[1],
#     }

# ── Config additions ────────────────────────────────────────────────────
N_SEEDS = 5          # number of CV repeats with different seeds
SEEDS = list(range(N_SEEDS))

def _cv_roc(X, y, groups_cv, label, n_splits=N_SPLITS, seeds=SEEDS):
    """
    Repeated CV with multiple seeds.
    Returns mean ROC + AUC stats aggregated across all (seed × fold) runs.
    """
    pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("rf", RandomForestClassifier(n_estimators=N_TREES,
                                       random_state=RANDOM_STATE, n_jobs=-1)),
    ])
    mean_fpr = np.linspace(0, 1, 100)
    tprs, aucs = [], []

    for seed in seeds:
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for train_idx, test_idx in cv.split(X, y, groups=groups_cv):
            pipe.fit(X[train_idx], y[train_idx])
            y_proba = pipe.predict_proba(X[test_idx])[:, 1]
            fpr, tpr, _ = roc_curve(y[test_idx], y_proba)
            interp_tpr = np.interp(mean_fpr, fpr, tpr)
            interp_tpr[0] = 0.0
            tprs.append(interp_tpr)
            aucs.append(auc(fpr, tpr))

    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    std_tpr = np.std(tprs, axis=0)
    return {
        "label": label,
        "mean_fpr": mean_fpr,
        "mean_tpr": mean_tpr,
        "std_tpr": std_tpr,
        "auc_mean": np.mean(aucs),
        "auc_std": np.std(aucs),
        "n_features": X.shape[1],
        "n_total_folds": len(aucs),
    }

# ── Run "All features" reference ────────────────────────────────────────
all_features = [f for feats in FEATURE_GROUPS.values() for f in feats]
print(f"Running 'All features' reference ({len(all_features)} features)...")
ref_result = _cv_roc(features_df_pruned[all_features].values, y, groups_cv, "All features")
print(f"  AUC = {ref_result['auc_mean']:.3f} ± {ref_result['auc_std']:.3f}")

# ── Run each group solo ─────────────────────────────────────────────────
group_results = {}
for gname, gfeats in FEATURE_GROUPS.items():
    print(f"Running group '{gname}' ({len(gfeats)} features)...")
    res = _cv_roc(features_df_pruned[gfeats].values, y, groups_cv, gname)
    print(f"  AUC = {res['auc_mean']:.3f} ± {res['auc_std']:.3f}")
    group_results[gname] = res

# ── Plot ────────────────────────────────────────────────────────────────
sorted_groups = sorted(group_results.items(),
                       key=lambda kv: kv[1]["auc_mean"], reverse=True)

fig, ax = plt.subplots(figsize=(8, 7))

# Reference: All features (plotted FIRST so it appears first in legend)
ax.plot(ref_result["mean_fpr"], ref_result["mean_tpr"], color="black",
        linewidth=2.5, linestyle="-",
        label=f"{'All features':<22s} AUC = {ref_result['auc_mean']:.3f} ± {ref_result['auc_std']:.3f}  (n={ref_result['n_features']})")
ax.fill_between(ref_result["mean_fpr"],
                 np.maximum(ref_result["mean_tpr"] - ref_result["std_tpr"], 0),
                 np.minimum(ref_result["mean_tpr"] + ref_result["std_tpr"], 1),
                 color="black", alpha=0.1)

# Individual groups (no std bands)
cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(FEATURE_GROUPS)))
for (gname, res), color in zip(sorted_groups, cmap):
    ax.plot(res["mean_fpr"], res["mean_tpr"], color=color, linewidth=1.5,
            label=f"{gname:<22s} AUC = {res['auc_mean']:.3f} ± {res['auc_std']:.3f}  (n={res['n_features']})")

# Diagonal (no legend entry)
ax.plot([0, 1], [0, 1], "k:", linewidth=0.6, alpha=0.5)

ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title(f"ROC by feature group ({N_SPLITS}-fold CV × {N_SEEDS} seeds, grouped by {GROUP_COL})")
ax.legend(loc="lower right", fontsize=8, frameon=False, prop={"family": "monospace"})
ax.grid(alpha=0.3, linestyle=":", linewidth=0.4)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)

plt.tight_layout()
plt.show()

# ── Summary table ──────────────────────────────────────────────────────
print("\n" + "═" * 70)
print(f"{'Group':<22s} {'N features':>12s} {'CV AUC':>20s}")
print("─" * 70)
for gname, res in sorted_groups:
    print(f"{gname:<22s} {res['n_features']:>12d}   {res['auc_mean']:.3f} ± {res['auc_std']:.3f}")
print("─" * 70)
print(f"{'All features':<22s} {ref_result['n_features']:>12d}   "
      f"{ref_result['auc_mean']:.3f} ± {ref_result['auc_std']:.3f}")

sdsfs


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Leave-one-group-out (LOGO) ablation:
#   for each group G, train on (all features EXCEPT G), measure AUC drop
# ═══════════════════════════════════════════════════════════════════════════
all_features = [f for feats in FEATURE_GROUPS.values() for f in feats]

# Baseline: full model (already computed as ref_result, but re-run for symmetry)
print(f"Running full-model baseline ({len(all_features)} features)...")
full_result = _cv_roc(features_df_pruned[all_features].values, y, groups_cv, "Full model")
print(f"  AUC = {full_result['auc_mean']:.3f} ± {full_result['auc_std']:.3f}")

# Ablation: drop each group in turn
logo_results = {}
for gname, gfeats in FEATURE_GROUPS.items():
    remaining = [f for f in all_features if f not in gfeats]
    print(f"Dropping '{gname}' ({len(gfeats)} features) → {len(remaining)} remaining...")
    res = _cv_roc(features_df_pruned[remaining].values, y, groups_cv,
                  f"− {gname}")
    delta = res["auc_mean"] - full_result["auc_mean"]
    print(f"  AUC = {res['auc_mean']:.3f} ± {res['auc_std']:.3f}  (Δ = {delta:+.3f})")
    logo_results[gname] = {**res, "delta": delta}

# ── Summary table ──────────────────────────────────────────────────────
# Sort by largest drop (most "needed" group at top)
sorted_logo = sorted(logo_results.items(), key=lambda kv: kv[1]["delta"])

print("\n" + "═" * 78)
print(f"{'Dropped group':<22s} {'N remain':>10s} {'AUC without':>15s} {'Δ vs full':>12s}")
print("─" * 78)
for gname, res in sorted_logo:
    auc_str = f"{res['auc_mean']:.3f} ± {res['auc_std']:.3f}"
    delta_str = f"{res['delta']:+.3f}"
    print(f"{gname:<22s} {res['n_features']:>10d}   {auc_str:>13s}   {delta_str:>10s}")
print("─" * 78)
print(f"{'(none — full model)':<22s} {full_result['n_features']:>10d}   "
      f"{full_result['auc_mean']:.3f} ± {full_result['auc_std']:.3f}        —")

# ── Plot: solo AUC vs LOGO drop, side by side per group ────────────────
fig, ax = plt.subplots(figsize=(10, 5.5))

# Use the order from solo ranking for easier comparison
solo_sorted = sorted(group_results.items(),
                     key=lambda kv: kv[1]["auc_mean"], reverse=True)
group_order = [g for g, _ in solo_sorted]

x = np.arange(len(group_order))
width = 0.38

solo_aucs = [group_results[g]["auc_mean"] for g in group_order]
solo_errs = [group_results[g]["auc_std"] for g in group_order]
drops = [-logo_results[g]["delta"] for g in group_order]   # flip sign: bigger = more needed
drop_errs = [logo_results[g]["auc_std"] for g in group_order]

# Left axis: solo AUC bars
b1 = ax.bar(x - width/2, solo_aucs, width, yerr=solo_errs,
            color="#4daf4a", edgecolor="black", linewidth=0.4,
            capsize=3, label="Solo AUC")
ax.set_ylabel("Solo CV AUC", color="#2d7a2d")
ax.tick_params(axis="y", labelcolor="#2d7a2d")
ax.axhline(0.5, color="gray", linestyle=":", linewidth=0.6, alpha=0.6)
ax.set_ylim(0.4, 1.0)

# Right axis: LOGO drop (loss when removed)
ax2 = ax.twinx()
b2 = ax2.bar(x + width/2, drops, width, yerr=drop_errs,
             color="#e41a1c", edgecolor="black", linewidth=0.4,
             capsize=3, label="AUC drop when removed")
ax2.set_ylabel("AUC drop when group is removed", color="#a01015")
ax2.tick_params(axis="y", labelcolor="#a01015")
ax2.axhline(0, color="gray", linestyle="-", linewidth=0.5, alpha=0.5)

ax.set_xticks(x)
ax.set_xticklabels(group_order, rotation=30, ha="right")
ax.set_title(f"Solo performance vs unique contribution per group "
             f"({N_SPLITS}-fold CV × {N_SEEDS} seeds)")

# Combined legend
lines = [b1, b2]
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, loc="upper right", frameon=False, fontsize=9)

for side in ("top",):
    ax.spines[side].set_visible(False)
    ax2.spines[side].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Individual permutation importance (per-feature), publication quality
# ═══════════════════════════════════════════════════════════════════════════
from sklearn.inspection import permutation_importance

# ── Config ──────────────────────────────────────────────────────────────
N_REPEATS = 20             # permutation repeats per fold
TOP_K = 20                 # how many features to show in the plot

# ── Build feature → group lookup ────────────────────────────────────────
feature_to_group = {f: g for g, feats in FEATURE_GROUPS.items() for f in feats}

# Distinct color per group (ColorBrewer-friendly; tweak as you like)
GROUP_COLORS = {
    "alphamissense":        "#e41a1c",
    "codon_usage":          "#377eb8",
    "physchem_deltas":      "#4daf4a",
    "physchem_wt":          "#984ea3",
    "variant_composition":  "#ff7f00",
    "variant_burden":       "#ffbf00",
    "rg_burden":            "#a65628",
    "rg_events":            "#f781bf",
    "substitution_classes": "#999999",
    "sequence_basic":       "#17becf",
}
# Fill in any missing groups with gray
for g in FEATURE_GROUPS:
    GROUP_COLORS.setdefault(g, "#666666")

# ── Compute permutation importance across seeds × folds ─────────────────
all_features = [f for feats in FEATURE_GROUPS.values() for f in feats]
X = features_df_pruned[all_features].values
feat_names = all_features

importances_per_fold = []   # list of arrays of shape (n_features,)

for seed in SEEDS:
    cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y, groups=groups_cv)):
        pipe = Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("rf", RandomForestClassifier(n_estimators=N_TREES,
                                           random_state=RANDOM_STATE, n_jobs=-1)),
        ])
        pipe.fit(X[train_idx], y[train_idx])
        result = permutation_importance(
            pipe, X[test_idx], y[test_idx],
            scoring="roc_auc", n_repeats=N_REPEATS,
            random_state=seed, n_jobs=-1,
        )
        importances_per_fold.append(result.importances_mean)
    print(f"Seed {seed} done ({len(importances_per_fold)} folds total)")

imp_matrix = np.vstack(importances_per_fold)   # shape (n_seeds*n_splits, n_features)
imp_mean = imp_matrix.mean(axis=0)
imp_std = imp_matrix.std(axis=0)

imp_df = pd.DataFrame({
    "feature": feat_names,
    "group": [feature_to_group[f] for f in feat_names],
    "importance_mean": imp_mean,
    "importance_std": imp_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

print("\nTop 15 features by permutation importance:")
print(imp_df.head(15).to_string(index=False))

# ── Plot ────────────────────────────────────────────────────────────────
top = imp_df.head(TOP_K).iloc[::-1]   # reverse for horizontal bar (top at top)

fig, ax = plt.subplots(figsize=(8, max(5, TOP_K * 0.22)))

colors = [GROUP_COLORS[g] for g in top["group"]]
y_pos = np.arange(len(top))

ax.barh(y_pos, top["importance_mean"], xerr=top["importance_std"],
        color=colors, edgecolor="black", linewidth=0.4,
        error_kw={"linewidth": 0.6, "ecolor": "#333333"})

ax.set_yticks(y_pos)
ax.set_yticklabels(top["feature"], fontsize=8)
ax.set_xlabel("Permutation importance (Δ ROC AUC when feature shuffled)")
ax.set_title(f"Top {TOP_K} features by permutation importance\n"
             f"({N_SPLITS}-fold CV × {N_SEEDS} seeds × {N_REPEATS} permutations)",
             fontsize=10)
ax.axvline(0, color="black", linewidth=0.5)
ax.grid(axis="x", alpha=0.3, linestyle=":", linewidth=0.4)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)

# Legend keyed to groups present in the top-K
groups_in_top = top["group"].unique()
legend_order = [g for g in FEATURE_GROUPS if g in groups_in_top]   # preserve dict order
from matplotlib.patches import Patch
handles = [Patch(facecolor=GROUP_COLORS[g], edgecolor="black",
                 linewidth=0.4, label=g) for g in legend_order]
ax.legend(handles=handles, loc="lower right", frameon=False, fontsize=8,
          title="Feature group", title_fontsize=8)

plt.tight_layout()
plt.show()

# Save the full sorted table for the supplement
imp_df.to_csv("permutation_importance_full.csv", index=False)
print(f"\nFull importance table saved (n={len(imp_df)} features).")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Quick RF classifier test: with AM only / without AM / with all features
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, roc_curve,
)
from sklearn.impute import SimpleImputer

# ── Setup ────────────────────────────────────────────────────────────────
RANDOM_STATE = 42
N_TREES = 100

EXCLUDE_COLS = {"region_id", "group", "label", "group_x", "group_y"}
AM_FEATURES = [c for c in features_df.columns
               if "alphamissense" in c.lower() or c.startswith("am_")
               or c == "fraction_pathogenic"]

print(f"Total features: {len(features_df.columns) - len(EXCLUDE_COLS)}")
print(f"AlphaMissense features ({len(AM_FEATURES)}): {AM_FEATURES}")
print(f"Dataset: {len(features_df)} regions "
      f"({(features_df['group'] == 'pos').sum()} pos, "
      f"{(features_df['group'] == 'neg').sum()} neg)")

y = (features_df["group"] == "pos").astype(int).values

def _run_rf(feature_subset, label):
    # Keep only numeric features
    X_df = features_df[feature_subset].select_dtypes(include=[np.number])
    dropped = set(feature_subset) - set(X_df.columns)
    if dropped:
        print(f"  (Dropped {len(dropped)} non-numeric features: {sorted(dropped)})")
    feature_subset = list(X_df.columns)
    X = X_df.values

    imputer = SimpleImputer(strategy="median")
    X_imputed = imputer.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_imputed, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
    )
    rf = RandomForestClassifier(
        n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    y_proba = rf.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_auc = cross_val_score(
        RandomForestClassifier(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1),
        X_imputed, y, scoring="roc_auc", cv=cv,
    )
    cv_acc = cross_val_score(
        RandomForestClassifier(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1),
        X_imputed, y, scoring="accuracy", cv=cv,
    )

    rf_full = RandomForestClassifier(
        n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf_full.fit(X_imputed, y)
    importances = pd.Series(
        rf_full.feature_importances_, index=feature_subset,
    ).sort_values(ascending=False)

    print(f"\n── {label} ────────────────────────────────────")
    print(f"  Features: {len(feature_subset)}")
    print(f"  Test set (80/20): accuracy = {acc:.3f}, AUC = {auc:.3f}")
    print(f"  5-fold CV:        accuracy = {cv_acc.mean():.3f} ± {cv_acc.std():.3f}, "
          f"AUC = {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")
    print(f"\n  Top 15 features by importance:")
    print(importances.head(15).to_string())

    return {
        "label": label,
        "test_accuracy": acc,
        "test_auc": auc,
        "cv_accuracy_mean": cv_acc.mean(),
        "cv_accuracy_std": cv_acc.std(),
        "cv_auc_mean": cv_auc.mean(),
        "cv_auc_std": cv_auc.std(),
        "importances": importances,
        "y_test": y_test,
        "y_proba": y_proba,
    }
# ── Run three configurations ────────────────────────────────────────────
all_features = [c for c in features_df.columns if c not in EXCLUDE_COLS]
features_without_am = [c for c in all_features if c not in AM_FEATURES]
features_only_am = list(AM_FEATURES)

result_all = _run_rf(all_features, "ALL features (with AM)")
result_without_am = _run_rf(features_without_am, "WITHOUT AlphaMissense")
result_only_am = _run_rf(features_only_am, "AM ONLY")


# ── Comparative visualization ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: ROC curves for all three configurations
ax = axes[0]
styles = [("-", "tab:blue"), ("--", "tab:green"), (":", "tab:orange")]
for r, (linestyle, color) in zip(
    [result_all, result_without_am, result_only_am], styles,
):
    fpr, tpr, _ = roc_curve(r["y_test"], r["y_proba"])
    ax.plot(fpr, tpr, linestyle=linestyle, color=color, linewidth=1.5,
            label=f"{r['label']} (AUC={r['test_auc']:.3f})")
ax.plot([0, 1], [0, 1], "k:", linewidth=0.6, alpha=0.5)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC comparison")
ax.legend(frameon=False, fontsize=9, loc="lower right")
ax.grid(alpha=0.3, linestyle=":", linewidth=0.4)

# Right: Top-20 feature importances (WITH AM)
ax = axes[1]
top20 = result_all["importances"].head(20)
colors = ["tab:orange" if f in AM_FEATURES else "tab:blue" for f in top20.index]
ax.barh(range(len(top20)), top20.values[::-1],
        color=colors[::-1], edgecolor="black", linewidth=0.3)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index[::-1], fontsize=7)
ax.set_xlabel("Feature importance")
ax.set_title("Top 20 features (ALL)  — orange = AM")
ax.grid(axis="x", alpha=0.3, linestyle=":", linewidth=0.4)

for side in ("top", "right"):
    axes[0].spines[side].set_visible(False)
    axes[1].spines[side].set_visible(False)

plt.tight_layout()
plt.show()

# ── Summary ─────────────────────────────────────────────────────────────
print("\n" + "═" * 60)
print("SUMMARY — 5-fold CV")
print("═" * 60)
print(f"{'Configuration':<30} {'CV AUC':>15} {'CV accuracy':>18}")
print("─" * 65)
for r in [result_all, result_without_am, result_only_am]:
    auc_str = f"{r['cv_auc_mean']:.3f} ± {r['cv_auc_std']:.3f}"
    acc_str = f"{r['cv_accuracy_mean']:.3f} ± {r['cv_accuracy_std']:.3f}"
    print(f"{r['label']:<30} {auc_str:>15} {acc_str:>18}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Leave-one-gene-out CV: honest generalization test
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
N_TREES = 100
EXCLUDE_COLS = {"region_id", "group", "label", "group_x", "group_y"}
AM_FEATURES = [c for c in features_df.columns
               if "alphamissense" in c.lower() or c.startswith("am_")]

# Extract gene (UniProt accession) from region_id
# region_id format: UniProtAccession_start_end
features_df = features_df.copy()
features_df["gene"] = features_df["region_id"].str.split("_").str[0]

n_genes = features_df["gene"].nunique()
region_per_gene = features_df["gene"].value_counts()
print(f"Number of unique genes: {n_genes}")
print(f"Regions per gene: min={region_per_gene.min()}, "
      f"median={region_per_gene.median():.0f}, "
      f"max={region_per_gene.max()}, "
      f"mean={region_per_gene.mean():.2f}")
print(f"Genes with >1 region: {(region_per_gene > 1).sum()}")

y = (features_df["group"] == "pos").astype(int).values
groups = features_df["gene"].values

def _run_logo(feature_subset, label):
    # Keep only numeric
    X_df = features_df[feature_subset].select_dtypes(include=[np.number])
    feature_subset = list(X_df.columns)
    X = X_df.values

    # Impute
    imputer = SimpleImputer(strategy="median")
    X_imputed = imputer.fit_transform(X)

    # Leave-one-gene-out CV
    logo = LeaveOneGroupOut()

    # Get per-region predictions via out-of-fold prediction
    rf = RandomForestClassifier(
        n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1,
    )
    y_proba = cross_val_predict(
        rf, X_imputed, y, groups=groups, cv=logo, method="predict_proba",
        n_jobs=-1,
    )[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    auc = roc_auc_score(y, y_proba)
    acc = accuracy_score(y, y_pred)

    print(f"\n── {label} ────────────────────────────────────")
    print(f"  Features: {len(feature_subset)}")
    print(f"  LOGO-CV accuracy: {acc:.3f}")
    print(f"  LOGO-CV AUC:      {auc:.3f}")

    return {
        "label": label,
        "auc": auc,
        "accuracy": acc,
        "y_true": y,
        "y_proba": y_proba,
    }


all_features = [c for c in features_df.columns
                if c not in EXCLUDE_COLS and c != "gene"]
features_without_am = [c for c in all_features if c not in AM_FEATURES]
features_only_am = list(AM_FEATURES)

result_all = _run_logo(all_features, "ALL features (with AM)")
result_without_am = _run_logo(features_without_am, "WITHOUT AlphaMissense")
result_only_am = _run_logo(features_only_am, "AM ONLY")


# ── ROC comparison figure ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
styles = [("-", "tab:blue"), ("--", "tab:green"), (":", "tab:orange")]
for r, (ls, color) in zip(
    [result_all, result_without_am, result_only_am], styles,
):
    fpr, tpr, _ = roc_curve(r["y_true"], r["y_proba"])
    ax.plot(fpr, tpr, linestyle=ls, color=color, linewidth=1.5,
            label=f"{r['label']} (AUC={r['auc']:.3f})")
ax.plot([0, 1], [0, 1], "k:", linewidth=0.6, alpha=0.5)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Leave-one-gene-out ROC")
ax.legend(frameon=False, fontsize=9, loc="lower right")
ax.grid(alpha=0.3, linestyle=":", linewidth=0.4)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()


# ── Final comparison vs earlier stratified 5-fold ──────────────────────
print("\n" + "═" * 60)
print("HONEST SUMMARY: stratified 5-fold vs leave-one-gene-out")
print("═" * 60)
print(f"{'Configuration':<30} {'5-fold AUC':>15} {'LOGO AUC':>15}")
print("─" * 60)
print(f"{'ALL features (with AM)':<30} {'0.838':>15} {result_all['auc']:>15.3f}")
print(f"{'WITHOUT AlphaMissense':<30} {'0.797':>15} {result_without_am['auc']:>15.3f}")
print(f"{'AM ONLY':<30} {'0.786':>15} {result_only_am['auc']:>15.3f}")
print("\nIf LOGO numbers are MUCH lower than 5-fold, the classifier was")
print("largely learning gene identity, not generalizable biology.")

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupKFold

# Use a single train/test split for permutation importance
# (LOGO with permutation across all 600 regions is slow; GroupKFold is cleaner)
gkf = GroupKFold(n_splits=5)
fold_importances = []

X_full = features_df[all_features].select_dtypes(include=[np.number])
feature_names = X_full.columns.tolist()
X_full = SimpleImputer(strategy="median").fit_transform(X_full.values)

for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(X_full, y, groups)):
    rf = RandomForestClassifier(
        n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf.fit(X_full[train_idx], y[train_idx])
    perm = permutation_importance(
        rf, X_full[test_idx], y[test_idx],
        n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1,
        scoring="roc_auc",
    )
    fold_importances.append(perm.importances_mean)

# Aggregate across folds
imp_df = pd.DataFrame(
    fold_importances, columns=feature_names,
).T
imp_df.columns = [f"fold_{i}" for i in range(len(fold_importances))]
imp_df["mean"] = imp_df.mean(axis=1)
imp_df["std"]  = imp_df[[f"fold_{i}" for i in range(len(fold_importances))]].std(axis=1)
imp_df = imp_df.sort_values("mean", ascending=False)

# ── Top 25 plot ────────────────────────────────────────────────────────
top = imp_df.head(25)
fig, ax = plt.subplots(figsize=(7, 8))
y_pos = np.arange(len(top))
ax.barh(y_pos, top["mean"], xerr=top["std"],
        color="tab:blue", alpha=0.7, edgecolor="black", linewidth=0.4)
ax.set_yticks(y_pos)
ax.set_yticklabels(top.index, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Permutation importance (Δ AUC)")
ax.set_title("Top 25 features (5-fold group permutation importance)")
ax.axvline(0, color="black", linewidth=0.5)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

# Optional: aggregate by feature category
def _category(f):
    if f.startswith("am_"):       return "AlphaMissense"
    if f.startswith("esm_"):      return "ESM1b LLR"
    if f.startswith("rg_"):       return "RG burden/events"
    if f.startswith("sub_rate_"): return "Substitution class"
    if f.startswith("delta_"):    return "Physchem Δ"
    if f.startswith("wt_"):       return "Physchem WT"
    if f.startswith("codon_"):    return "Codon usage"
    if f.startswith("density_") or f.startswith("fraction_") or f.startswith("n_"):
        return "Variant burden"
    return "Other"

imp_df["category"] = [_category(f) for f in imp_df.index]
cat_summary = (
    imp_df.groupby("category")["mean"]
          .agg(["sum", "mean", "count"])
          .sort_values("sum", ascending=False)
)
print("\nFeature category contribution to importance:")
print(cat_summary)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Feature importance: permutation vs tree-based (MDI), side by side, 
# colored by category
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer

# ── Category palette ──────────────────────────────────────────────────────
# Consistent across both panels for direct comparison
CATEGORY_COLORS = {
    "AlphaMissense":       "#e41a1c",  # red
    "ESM1b LLR":           "#984ea3",  # purple
    "RG burden/events":    "#4daf4a",  # green
    "Substitution class":  "#377eb8",  # blue
    "Physchem Δ":          "#ff7f00",  # orange
    "Physchem WT":         "#ffcc00",  # yellow-orange
    "Codon usage":         "#a65628",  # brown
    "Variant burden":      "#f781bf",  # pink
    "Other":               "#999999",  # grey
}

def _category(f):
    if f.startswith("am_"):       return "AlphaMissense"
    if f.startswith("esm_"):      return "ESM1b LLR"
    if f.startswith("rg_"):       return "RG burden/events"
    if f.startswith("sub_rate_"): return "Substitution class"
    if f.startswith("delta_"):    return "Physchem Δ"
    if f.startswith("wt_"):       return "Physchem WT"
    if f.startswith("codon_"):    return "Codon usage"
    if (f.startswith("density_") or f.startswith("fraction_") 
        or f.startswith("n_")):
        return "Variant burden"
    return "Other"


# ── Prep features ──────────────────────────────────────────────────────────
X_full_df = features_df[all_features].select_dtypes(include=[np.number])
feature_names = X_full_df.columns.tolist()
imputer = SimpleImputer(strategy="median")
X_full = imputer.fit_transform(X_full_df.values)


# ── Permutation importance (5-fold group, mean across folds) ──────────────
print("Computing permutation importance...")
gkf = GroupKFold(n_splits=5)
fold_perm = []
for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(X_full, y, groups)):
    rf_fold = RandomForestClassifier(
        n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf_fold.fit(X_full[train_idx], y[train_idx])
    perm = permutation_importance(
        rf_fold, X_full[test_idx], y[test_idx],
        n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1,
        scoring="roc_auc",
    )
    fold_perm.append(perm.importances_mean)
    print(f"  fold {fold_idx+1}/5 done")

perm_imp_mean = np.mean(fold_perm, axis=0)
perm_imp_std = np.std(fold_perm, axis=0)

perm_df = pd.DataFrame({
    "feature": feature_names,
    "importance_mean": perm_imp_mean,
    "importance_std":  perm_imp_std,
    "category":        [_category(f) for f in feature_names],
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)


# ── Tree (MDI) importance — single model fit on all data ──────────────────
print("\nComputing tree-based importance (MDI)...")
rf_full = RandomForestClassifier(
    n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1,
)
rf_full.fit(X_full, y)
mdi_imp = rf_full.feature_importances_

# For uncertainty, also collect per-tree MDI std across the forest
mdi_per_tree = np.array([
    tree.feature_importances_ for tree in rf_full.estimators_
])
mdi_std = mdi_per_tree.std(axis=0)

mdi_df = pd.DataFrame({
    "feature":         feature_names,
    "importance_mean": mdi_imp,
    "importance_std":  mdi_std,
    "category":        [_category(f) for f in feature_names],
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)


# ── Plot: side-by-side, top 25, colored by category ───────────────────────
TOP_N = 25
top_perm = perm_df.head(TOP_N)
top_mdi  = mdi_df.head(TOP_N)

fig, axes = plt.subplots(1, 2, figsize=(13, 9))

def _plot_panel(ax, df_top, title, xlabel):
    y_pos = np.arange(len(df_top))
    colors = [CATEGORY_COLORS[c] for c in df_top["category"]]
    ax.barh(
        y_pos, df_top["importance_mean"],
        xerr=df_top["importance_std"],
        color=colors, alpha=0.85, edgecolor="black", linewidth=0.5,
        error_kw=dict(ecolor="black", lw=0.6, capsize=2),
    )
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_top["feature"], fontsize=7.5)
    ax.invert_yaxis()
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.axvline(0, color="black", linewidth=0.5)
    ax.grid(alpha=0.3, linestyle=":", linewidth=0.4, axis="x")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

_plot_panel(
    axes[0], top_perm,
    title="Permutation importance (5-fold group)",
    xlabel="Δ AUC when permuted",
)
_plot_panel(
    axes[1], top_mdi,
    title="Tree-based importance (MDI)",
    xlabel="Mean decrease in impurity",
)

# Shared category legend at the bottom
present_categories = sorted(set(
    list(top_perm["category"]) + list(top_mdi["category"])
))
legend_handles = [
    plt.Rectangle((0,0), 1, 1, facecolor=CATEGORY_COLORS[c],
                   edgecolor="black", linewidth=0.5, label=c)
    for c in present_categories
]
fig.legend(
    handles=legend_handles,
    loc="lower center", ncol=min(len(present_categories), 5),
    frameon=False, fontsize=8.5,
    bbox_to_anchor=(0.5, -0.01),
)

fig.suptitle(
    "Feature importance for the classifier (top 25)",
    fontsize=12, y=0.995,
)
plt.tight_layout(rect=[0, 0.04, 1, 0.985])
plt.savefig(
    "/mnt/d/phd/scripts/16_ev_signature_predictor/figures/"
    "feature_importance_comparison.svg",
    dpi=600, bbox_inches="tight",
)
plt.savefig(
    "/mnt/d/phd/scripts/16_ev_signature_predictor/figures/"
    "feature_importance_comparison.png",
    dpi=600, bbox_inches="tight",
)
plt.show()


# ── Category-level summary ────────────────────────────────────────────────
print("\n" + "═" * 60)
print("Category contributions (full feature list, not just top 25)")
print("═" * 60)

cat_summary_perm = (
    perm_df.groupby("category")["importance_mean"]
    .agg(["sum", "mean", "count"])
    .sort_values("sum", ascending=False)
)
cat_summary_mdi = (
    mdi_df.groupby("category")["importance_mean"]
    .agg(["sum", "mean", "count"])
    .sort_values("sum", ascending=False)
)

print("\nPermutation importance — by category:")
print(cat_summary_perm.round(4))
print("\nTree importance (MDI) — by category:")
print(cat_summary_mdi.round(4))


# ── Optional: rank correlation between the two methods ────────────────────
from scipy.stats import spearmanr
both = perm_df[["feature", "importance_mean"]].rename(
    columns={"importance_mean": "perm"}
).merge(
    mdi_df[["feature", "importance_mean"]].rename(
        columns={"importance_mean": "mdi"}
    ),
    on="feature",
)
rho, p = spearmanr(both["perm"], both["mdi"])
print(f"\nSpearman correlation between permutation and MDI rankings: "
      f"ρ = {rho:.3f}, p = {p:.2e}")
print("(High ρ = both methods agree on which features matter)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Save trained classifiers for inference on new data
# ═══════════════════════════════════════════════════════════════════════════
import joblib
import sklearn
from datetime import datetime
from pathlib import Path

MODELS_DIR = Path("/mnt/d/phd/scripts/16_ev_signature_predictor/models")
MODELS_DIR.mkdir(exist_ok=True, parents=True)

def _train_and_save(feature_subset, label, filename):
    """
    Fit final RandomForest on ALL data (no holdout — already evaluated via LOGO)
    and save to disk along with all metadata needed for inference.
    """
    X_df = features_df[feature_subset].select_dtypes(include=[np.number])
    feature_subset_final = list(X_df.columns)
    X = X_df.values

    # Fit imputer on training data (same strategy as in _run_logo)
    imputer = SimpleImputer(strategy="median")
    X_imputed = imputer.fit_transform(X)

    # Fit final RandomForest on all data
    rf = RandomForestClassifier(
        n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf.fit(X_imputed, y)

    out_path = MODELS_DIR / filename
    joblib.dump({
        "model": rf,
        "imputer": imputer,
        "feature_names": feature_subset_final,    # in TRAINING ORDER
        "label": label,
        "training_date": datetime.now().isoformat(),
        "sklearn_version": sklearn.__version__,
        "n_train_samples": int(len(y)),
        "n_features": len(feature_subset_final),
        "training_metadata": {
            "n_trees": N_TREES,
            "random_state": RANDOM_STATE,
            "exclude_cols": list(EXCLUDE_COLS),
            "label_definition": "1 = pos, 0 = neg",
        },
    }, out_path)
    print(f"  Saved {label} → {out_path}")
    return out_path


print("Saving trained classifiers for future inference...")
_train_and_save(all_features,        "ALL_with_AM",       "classifier_all_features.joblib")
_train_and_save(features_without_am, "WITHOUT_AM",        "classifier_without_am.joblib")
_train_and_save(features_only_am,    "AM_ONLY",           "classifier_am_only.joblib")
print(f"\nAll classifiers saved to {MODELS_DIR}/")